# Traceprop-LLM — 2-GPU DDP overhead (item 11, optional/bonus)

**Before running:** Kaggle notebook settings (right sidebar) → Accelerator → **GPU T4 x2**. This is Kaggle-specific; Colab has no multi-GPU runtime tier at all.

**Claim being tested:** inline capture hooks into each DDP rank's *local* backward pass and writes to a *local* buffer — no extra cross-rank communication beyond DDP's own (unavoidable) gradient all-reduce. If true, per-rank overhead here should land close to the single-GPU Table 1 number for the same model, not scale up with world size, and each rank's logged sample indices should be disjoint (nothing gathered).

This is marked optional in the paper's Limitations already — a positive result strengthens item 11, a negative one just means the caveat stays as-is.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python --version

In [ ]:
!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2"
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), 'device count', torch.cuda.device_count())

## Get the repo

Kaggle notebooks don't have an interactive `getpass` prompt the same way Colab does in all contexts, but it works fine in the standard notebook editor. If it doesn't, add your token as a Kaggle Secret instead (Add-ons → Secrets) and read it via `from kaggle_secrets import UserSecretsClient`.

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /kaggle/working/Traceprop || (cd /kaggle/working/Traceprop && git pull -q)
%cd /kaggle/working/Traceprop
!pip -q install -e .

## Run: 2-GPU DDP overhead

`torchrun` launches one process per GPU; each rank writes its own `results/exp33_ddp_rank{R}.json`, then rank 0 aggregates. Uses the same batch/seq/repeats config as Table 1 (batch=16, seq=64, repeats=20) so the per-rank number is directly comparable to the single-GPU GPT-2 row (1.08±0.14%).

In [ ]:
%cd /kaggle/working/Traceprop/experiments
!torchrun --standalone --nproc_per_node=2 exp33_ddp_overhead.py \
    --backend hf --model gpt2 --steps 200 --warmup 10 --repeats 20 \
    --batch 16 --seq 64 --rank 8 --proj_dim 512 --track 1

## Result

Check `results/exp33_ddp_summary.json`: `per_rank_overhead_pct_median` should be close to Table 1's GPT-2 throughput number (1.08±0.14%) and close to each other across ranks, and `logged_index_ranges_disjoint_across_ranks` should be `true`. If overhead is noticeably higher than single-GPU, that's a real finding too — report it rather than reach for an explanation, and bring the numbers back before writing anything into the paper.

In [ ]:
import json
print(json.dumps(json.load(open('results/exp33_ddp_summary.json')), indent=2))